In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
# Carpeta donde están los datos originales del INEI (módulos ENAHO)
datos_originales = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO"

# Carpeta donde se guardarán los resultados (MASTER_*, PANEL_*, BASE_REGRESIONES)
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

# Crear carpeta de resultados si no existe
os.makedirs(base_resultados, exist_ok=True)

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def encontrar_archivo(carpeta, inicio_nombre):
    for f in os.listdir(carpeta):
        if f.upper().replace(".CSV", "").startswith(inicio_nombre.upper()):
            return os.path.join(carpeta, f)
    raise FileNotFoundError(f"No se encontró {inicio_nombre} en {carpeta}")

def leer_csv_inei(ruta):
    for sep in [",", ";"]:
        for enc in ["utf-8-sig", "latin1"]:
            try:
                df = pd.read_csv(ruta, sep=sep, encoding=enc, low_memory=False, dtype=str)
                if len(df.columns) > 1:
                    df.columns = df.columns.str.strip()
                    return df
            except Exception:
                pass
    raise ValueError(f"No se pudo leer el archivo: {ruta}")

def to_num(serie):
    return pd.to_numeric(serie, errors="coerce")

def construir_llave_persona(df, llaves=None, nombre="llave_persona"):
    if llaves is None:
        llaves = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
    df = df.copy()
    for c in llaves:
        df[f"_{c}_k"] = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype(str)
    df[nombre] = df[[f"_{c}_k" for c in llaves]].agg("-".join, axis=1)
    return df.drop(columns=[f"_{c}_k" for c in llaves])

# Preguntas P558H{1..12}_7: "medio de pago usado = billetera digital (Yape, Plin, etc.)"
# para cada una de las 12 categorías de gasto (alimentos, servicios de vivienda,
# combustible, aseo, vestido, muebles, electrodomésticos, 3x "otro").
# Código: 0 = Pase (no marcó esa opción) / 7 = Billetera digital (sí marcó)
COLS_USO_BILLETERA = [f"P558H{i}_7" for i in range(1, 13)]

def reparar_variables(df):
    df = df.copy()

    df["Edad"] = to_num(df["P208A"]) if "P208A" in df.columns else df.get("Edad")
    df["P507_num"] = to_num(df["P507"])
    df["P510A1_num"] = to_num(df["P510A1"])
    df["P511A_num"] = to_num(df["P511A"])

    # --- FIX 1: Ocupado ---
    df["Ocupado"] = df["P507_num"].notna().astype(int)

    # --- FIX 3: sin_sunat restringido a P507==2 ---
    tfnr = df["P507_num"] == 5
    trab_hogar = df["P507_num"] == 6
    sin_sunat = (df["P507_num"] == 2) & (df["P510A1_num"] == 3)
    dependiente = df["P507_num"].isin([3, 4])
    sin_contrato = df["P511A_num"] == 7

    df["Informal"] = (
        sin_sunat.fillna(False)
        | tfnr.fillna(False)
        | trab_hogar.fillna(False)
        | (dependiente & sin_contrato.fillna(False))
    ).astype(int)
    df.loc[df["Ocupado"] == 0, "Informal"] = pd.NA

    # --- Crédito formal ---
    df["P558E1_4"] = to_num(df.get("P558E1_4")).fillna(0)
    df["P558E1_9"] = to_num(df.get("P558E1_9")).fillna(0)
    df["CreditoFormal"] = ((df["P558E1_4"] == 4) | (df["P558E1_9"] == 9)).astype(int)

    # --- TenenciaBilletera (antes "Billetera") ---
    # P558E1_8: ¿Tiene billetera digital como producto financiero? (0=Pase, 8=Sí tiene)
    df["P558E1_8"] = to_num(df.get("P558E1_8")).fillna(0)
    df["TenenciaBilletera"] = (df["P558E1_8"] == 8).astype(int)

    # --- UsoBilletera (nueva) ---
    # 1 si marcó billetera digital como medio de pago en AL MENOS UNA
    # de las 12 categorías de gasto (P558H1_7 ... P558H12_7)
    cols_uso_presentes = [c for c in COLS_USO_BILLETERA if c in df.columns]
    if cols_uso_presentes:
        uso_flags = pd.concat(
            [to_num(df[c]).fillna(0) == 7 for c in cols_uso_presentes], axis=1
        )
        df["UsoBilletera"] = uso_flags.any(axis=1).astype(int)
    else:
        df["UsoBilletera"] = pd.NA

    # --- FIX 4: CreditoInformal con P558G7 ---
    if "P558G7" in df.columns:
        df["CreditoInformal"] = (to_num(df["P558G7"]).fillna(0) == 7).astype(int)

    return df

def construir_master(anio, carpetas):
    print(f"Cargando módulos {anio}...")

    ruta_200 = encontrar_archivo(*carpetas["Mod200"])
    df200 = leer_csv_inei(ruta_200)[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P203", "P207", "P208A"]]

    ruta_300 = encontrar_archivo(*carpetas["Mod300"])
    df300 = leer_csv_inei(ruta_300)[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P301A", "P301B"]]

    ruta_500 = encontrar_archivo(*carpetas["Mod500"])
    df500_full = leer_csv_inei(ruta_500)
    cols_500 = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO",
                "P507", "P510A1", "P511A", "P512A",
                "P558E1_4", "P558E1_8", "P558E1_9", "P558G7"] + COLS_USO_BILLETERA
    df500 = df500_full[[c for c in cols_500 if c in df500_full.columns]]

    ruta_100 = encontrar_archivo(*carpetas["Mod100"])
    df100 = leer_csv_inei(ruta_100)
    cols_100 = ["CONGLOME", "VIVIENDA", "HOGAR", "DOMINIO", "ESTRATO", "PANEL"]
    for fac in ["FACTOR", "FACPOB", "FACTOR07"]:
        if fac in df100.columns:
            cols_100.append(fac)
            break
    df100 = df100[cols_100]

    ruta_sum = encontrar_archivo(*carpetas["Sumaria"])
    df_sum = leer_csv_inei(ruta_sum)
    cols_sum = ["CONGLOME", "VIVIENDA", "HOGAR"]
    gasto_encontrada = None
    for g in ["GASHOG2D", "GASHOG1D", "GASHOG2", "GASHOG1"]:
        if g in df_sum.columns:
            gasto_encontrada = g
            break
    if gasto_encontrada:
        df_sum = df_sum.rename(columns={gasto_encontrada: "GASHOG2"})
        cols_sum.append("GASHOG2")
    for m in ["MIEPERHO", "MIEMBRO"]:
        if m in df_sum.columns:
            cols_sum.append(m)
            break
    df_sum = df_sum[cols_sum]

    llave_persona = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
    llave_hogar = ["CONGLOME", "VIVIENDA", "HOGAR"]

    df = df200.merge(df300, on=llave_persona, how="left")
    df = df.merge(df500, on=llave_persona, how="left")
    df = df.merge(df100, on=llave_hogar, how="left")
    df = df.merge(df_sum, on=llave_hogar, how="left")

    df = construir_llave_persona(df)
    df = reparar_variables(df)

    ruta_salida = os.path.join(base_resultados, f"MASTER_{anio}.csv")
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"✅ Guardado: {ruta_salida}  ({df.shape[0]:,} x {df.shape[1]})")
    return df

# ============================================================
# 1. CONSTRUIR MASTER 2024 y 2025
# ============================================================

carpetas_2024 = {
    "Mod100": (os.path.join(datos_originales, "966-Modulo01"), "Enaho01-2024-100"),
    "Mod200": (os.path.join(datos_originales, "966-Modulo02"), "Enaho01-2024-200"),
    "Mod300": (os.path.join(datos_originales, "966-Modulo03"), "Enaho01A-2024-300"),
    "Mod500": (os.path.join(datos_originales, "966-Modulo05"), "Enaho01a-2024-500"),
    "Sumaria": (os.path.join(datos_originales, "966-Modulo34"), "Sumaria-2024-12g"),
}
df_master_2024 = construir_master("2024", carpetas_2024)

carpetas_2025 = {
    "Mod100": (os.path.join(datos_originales, "1031-Modulo01-2025"), "Enaho01-2025-100"),
    "Mod200": (os.path.join(datos_originales, "1031-Modulo02-2025"), "Enaho01-2025-200"),
    "Mod300": (os.path.join(datos_originales, "1031-Modulo03-2025"), "Enaho01A-2025-300"),
    "Mod500": (os.path.join(datos_originales, "1031-Modulo05-2025"), "Enaho01a-2025-500"),
    "Sumaria": (os.path.join(datos_originales, "1031-Modulo34-2025"), "Sumaria-2025-12g"),
}
df_master_2025 = construir_master("2025", carpetas_2025)

# ============================================================
# 2. CONSTRUIR PANEL (match por llave_persona entre 2024 y 2025)
# ============================================================

df24 = pd.read_csv(os.path.join(base_resultados, "MASTER_2024.csv"), encoding="utf-8-sig", low_memory=False)
df25 = pd.read_csv(os.path.join(base_resultados, "MASTER_2025.csv"), encoding="utf-8-sig", low_memory=False)

df24 = construir_llave_persona(df24)
df25 = construir_llave_persona(df25)

# --- Panel definido solo por coincidencia de llave_persona entre 2024 y 2025 ---
# (ya no se usa la variable PANEL: en el archivo 2024 esa pregunta se refiere a
# si el hogar fue entrevistado en 2023, no dice nada sobre su continuidad en 2025)
match = df24[["llave_persona"]].drop_duplicates().merge(
    df25[["llave_persona"]].drop_duplicates(), on="llave_persona", how="inner"
)

cols_2024 = ["llave_persona", "P203", "P207", "Edad", "Ocupado", "Informal",
             "TenenciaBilletera", "UsoBilletera", "CreditoFormal", "FACTOR07", "P301A",
             "ESTRATO", "DOMINIO", "MIEPERHO", "CONGLOME"]
df_2024_sub = df24[df24["llave_persona"].isin(match["llave_persona"])][cols_2024].rename(
    columns={"CreditoFormal": "CreditoFormal_2024"}
)

cols_2025 = ["llave_persona", "CreditoFormal"]
if "CreditoInformal" in df25.columns:
    cols_2025.append("CreditoInformal")
df_2025_sub = df25[cols_2025].rename(
    columns={"CreditoFormal": "CreditoFormal_2025", "CreditoInformal": "CreditoInformal_2025"}
)

panel_df = df_2024_sub.merge(df_2025_sub, on="llave_persona", how="left")
panel_df.to_csv(os.path.join(base_resultados, "PANEL_2024_2025.csv"), index=False, encoding="utf-8-sig")
print(f"✅ PANEL_2024_2025.csv guardado ({panel_df.shape[0]:,} filas)")

# ============================================================
# 3. FILTRAR MUESTRA ANALÍTICA
# ============================================================

df = pd.read_csv(os.path.join(base_resultados, "PANEL_2024_2025.csv"), encoding="utf-8-sig", low_memory=False)

df["P203_num"] = pd.to_numeric(df["P203"], errors="coerce")
df["Ocupado_num"] = pd.to_numeric(df["Ocupado"], errors="coerce")
df["CreditoFormal_2024_num"] = pd.to_numeric(df["CreditoFormal_2024"], errors="coerce")
df["CreditoFormal_2025_num"] = pd.to_numeric(df["CreditoFormal_2025"], errors="coerce")
df["Informal_num"] = pd.to_numeric(df["Informal"], errors="coerce")

df = df[df["P203_num"] == 1]                     # jefes de hogar
df = df[df["Ocupado_num"] == 1]                  # ocupados (corregido)
df = df[df["CreditoFormal_2024_num"] == 0]       # sin crédito formal en 2024
df = df.dropna(subset=["Informal_num", "TenenciaBilletera", "UsoBilletera", "P207", "Edad", "P301A"])

df["NuevoCredito"] = (df["CreditoFormal_2025_num"] == 1).astype(int)

df.to_csv(os.path.join(base_resultados, "BASE_REGRESIONES.csv"), index=False, encoding="utf-8-sig")
print(f"✅ BASE_REGRESIONES.csv guardado ({df.shape[0]:,} filas)")

print("\n" + "="*60)
print("PROCESO COMPLETADO. ARCHIVOS GENERADOS:")
print("="*60)
print(f"📁 {os.path.join(base_resultados, 'MASTER_2024.csv')}")
print(f"📁 {os.path.join(base_resultados, 'MASTER_2025.csv')}")
print(f"📁 {os.path.join(base_resultados, 'PANEL_2024_2025.csv')}")
print(f"📁 {os.path.join(base_resultados, 'BASE_REGRESIONES.csv')}")

Cargando módulos 2024...
✅ Guardado: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER_2024.csv  (117,721 x 46)
Cargando módulos 2025...
